# Invoice Extraction Scorecard

This notebook validates the outputs of different prompt and LLM model experiments for invoice data extraction.

## Features
- Random sampling of PDFs from `raw.email_download_queue` with reproducible seed
- Automatic S3 download with local caching
- Load PDFs from a local directory
- Download specific PDFs by S3 keys
- Gemini model extraction using existing methods
- Validation against PDF content (hallucination detection)
- Scorecard visualization of results

## PDF Loading Methods
1. **`get_pdfs()`** - Random sampling from database with vendor diversity
2. **`get_pdfs_by_keys(s3_keys)`** - Download specific PDFs by S3 key
3. **`get_pdfs_from_local(directory)`** - Load PDFs from a local directory

In [62]:
!uv pip install pdf2image weasyprint

Using Python 3.12.3 environment at: /var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/.venv
Resolved 14 packages in 537ms                                        
Prepared 9 packages in 777ms                                             
Uninstalled 1 package in 4ms
Installed 9 packages in 18ms                                
 + brotli==1.2.0
 + cssselect2==0.8.0
 + fonttools==4.61.1
 + pydyf==0.12.1
 + pyphen==0.17.2
 - tinycss2==1.4.0
 + tinycss2==1.5.1
 + tinyhtml5==2.0.0
 + weasyprint==68.0
 + zopfli==0.4.0


In [1]:
# if you need hot-reload
import importlib                                                                                                                                                                                                                                                                                     
import data_extraction                                                                                                                                                                                                                                                                               
importlib.reload(data_extraction)

<module 'data_extraction' from '/var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/scripts/backfill_invoices_data/data_extraction.py'>

In [2]:
# Imports
import os
import sys
import json
import random
import time
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
from IPython.display import display, HTML

# Ensure the backfill_invoices_data directory is in the path
script_dir = Path.cwd()
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

# Import existing modules
from data_extraction import (
    extract_data_using_gemini,
    GEMINI_SYSTEM_PROMPT,
    GEMINI_USER_PROMPT_TEMPLATE
)
from data_validation import validate_header_data, validate_line_items_data
from pdf_utils import normalize_pdf_text
from invoice_extraction_schema import InvoiceExtractionSchema
from enums import GEMINI_MODEL
from db_utils import connect_to_db
from s3 import download_file_from_s3
from psycopg2.extras import RealDictCursor

print(f"Loaded prompts from data_extraction.py")
print(f"  System prompt: {len(GEMINI_SYSTEM_PROMPT)} chars")
print(f"  User prompt template: {len(GEMINI_USER_PROMPT_TEMPLATE)} chars")

Loaded prompts from data_extraction.py
  System prompt: 4289 chars
  User prompt template: 1994 chars


In [3]:
# Configuration
DEFAULT_SEED = 42
DEFAULT_SAMPLE_SIZE = 5

# Local cache directory for downloaded PDFs (within the repo)
CACHE_DIR = Path("temp_pdfs")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF cache directory: {CACHE_DIR.resolve()}")

PDF cache directory: /var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/scripts/backfill_invoices_data/temp_pdfs


In [4]:
def get_all_vendors(status_filter: str = "DOWNLOADED") -> List[str]:
    """
    Get all distinct vendors (senders) from email_download_queue that have PDFs.
    
    Args:
        status_filter: Filter by email status (default: "DOWNLOADED").
    
    Returns:
        List[str]: List of distinct sender email addresses.
    """
    connection = connect_to_db()
    if connection is None:
        raise ConnectionError("Failed to connect to database")
    
    cursor = connection.cursor(cursor_factory=RealDictCursor)
    
    try:
        query = """
            SELECT DISTINCT sender
            FROM raw.email_download_queue
            WHERE status = %s
              AND s3_keys IS NOT NULL
              AND jsonb_array_length(s3_keys) > 0
              AND deleted_at IS NULL
            ORDER BY sender
        """
        cursor.execute(query, (status_filter,))
        results = cursor.fetchall()
        return [row['sender'] for row in results]
    finally:
        cursor.close()
        connection.close()


def get_pdfs_from_db(
    sample_size: int = DEFAULT_SAMPLE_SIZE,
    seed: int = DEFAULT_SEED,
    status_filter: str = "DOWNLOADED",
    ensure_all_vendors: bool = True
) -> List[Dict[str, Any]]:
    """
    Get a random sample of PDF records from raw.email_download_queue.
    
    Queries the database for emails with PDFs and returns metadata
    including S3 keys for downloading. When ensure_all_vendors=True,
    guarantees at least one PDF from each distinct vendor (sender).
    
    Args:
        sample_size: Number of PDF records to return.
        seed: Random seed for reproducible sampling.
        status_filter: Filter by email status (default: "DOWNLOADED").
        ensure_all_vendors: If True, ensure at least one PDF from each vendor
            is included before filling with random samples.
    
    Returns:
        List[Dict]: List of dicts with keys:
            - id: email_download_queue UUID
            - sender: email sender
            - subject: email subject
            - s3_keys: list of S3 keys for PDFs
            - email_date: when the email was received
    """
    connection = connect_to_db()
    if connection is None:
        raise ConnectionError("Failed to connect to database")
    
    cursor = connection.cursor(cursor_factory=RealDictCursor)
    
    try:
        # Set seed for reproducibility
        cursor.execute("SELECT setseed(%s)", (seed / 2147483647.0,))
        
        if ensure_all_vendors:
            # Step 1: Get one random PDF from each vendor
            # Uses DISTINCT ON to get one row per sender, ordered randomly
            vendor_query = """
                SELECT DISTINCT ON (sender)
                    id,
                    sender,
                    subject,
                    s3_keys,
                    email_date,
                    in_email_pdf_count
                FROM raw.email_download_queue
                WHERE status = %s
                  AND s3_keys IS NOT NULL
                  AND jsonb_array_length(s3_keys) > 0
                  AND deleted_at IS NULL
                ORDER BY sender, RANDOM()
            """
            cursor.execute(vendor_query, (status_filter,))
            vendor_records = cursor.fetchall()
            
            num_vendors = len(vendor_records)
            print(f"Found {num_vendors} distinct vendors")
            
            if num_vendors == 0:
                print(f"No records found with status={status_filter}")
                return []
            
            # Convert to list of dicts
            records = [dict(row) for row in vendor_records]
            
            # Step 2: If we need more samples, get additional random PDFs
            if sample_size > num_vendors:
                remaining = sample_size - num_vendors
                # Get IDs we already have to exclude them (cast to UUID array)
                existing_ids = tuple(r['id'] for r in records)
                
                additional_query = """
                    SELECT 
                        id,
                        sender,
                        subject,
                        s3_keys,
                        email_date,
                        in_email_pdf_count
                    FROM raw.email_download_queue
                    WHERE status = %s
                      AND s3_keys IS NOT NULL
                      AND jsonb_array_length(s3_keys) > 0
                      AND deleted_at IS NULL
                      AND id NOT IN %s
                    ORDER BY RANDOM()
                    LIMIT %s
                """
                cursor.execute(additional_query, (status_filter, existing_ids, remaining * 2))
                additional_records = cursor.fetchall()
                records.extend([dict(row) for row in additional_records])
            
            # Shuffle to mix vendors (reproducibly)
            random.seed(seed)
            random.shuffle(records)
            
            return records[:sample_size]
        
        else:
            # Original behavior: pure random sampling
            query = """
                SELECT 
                    id,
                    sender,
                    subject,
                    s3_keys,
                    email_date,
                    in_email_pdf_count
                FROM raw.email_download_queue
                WHERE status = %s
                  AND s3_keys IS NOT NULL
                  AND jsonb_array_length(s3_keys) > 0
                  AND deleted_at IS NULL
                ORDER BY RANDOM()
                LIMIT %s
            """
            cursor.execute(query, (status_filter, sample_size * 3))
            results = cursor.fetchall()
            
            if not results:
                print(f"No records found with status={status_filter}")
                return []
            
            return [dict(row) for row in results][:sample_size]
        
    finally:
        cursor.close()
        connection.close()

In [5]:
def download_pdf_if_needed(s3_key: str, cache_dir: Path = CACHE_DIR) -> Path | None:
    """
    Download a PDF from S3 if not already cached locally.
    
    Args:
        s3_key: S3 key for the PDF file.
        cache_dir: Local directory for caching downloads.
    
    Returns:
        Path: Local path to the PDF file, or None if download failed.
    """
    # Create a safe local filename from the S3 key
    # Replace path separators with underscores to flatten the structure
    safe_filename = s3_key.replace("/", "_").replace("\\", "_")
    local_path = cache_dir / safe_filename
    
    # Check if already cached
    if local_path.exists():
        return local_path
    
    # Download from S3
    downloaded_path = download_file_from_s3(s3_key)
    
    if downloaded_path is None:
        return None
    
    # Move to cache directory with safe filename
    downloaded = Path(downloaded_path)
    if downloaded.exists():
        # Copy to cache location (keep original in invoices/ for compatibility)
        import shutil
        shutil.copy2(downloaded, local_path)
        return local_path
    
    return None

In [6]:
def get_pdfs_by_keys(
    s3_keys: List[str],
    cache_dir: Path = CACHE_DIR
) -> List[Dict[str, Any]]:
    """
    Download specific PDFs by their S3 keys.

    Use this when you want to test specific invoices rather than random sampling.
    Downloads PDFs from S3 (caching locally) and returns metadata with local paths.

    Args:
        s3_keys: List of S3 keys to download (e.g.,
            ["invoices/billing@vendor.com/2024-01/invoice.pdf", ...])
        cache_dir: Local directory for caching downloaded PDFs.

    Returns:
        List[Dict]: List of dicts with keys:
            - local_path: Path to the local PDF file
            - s3_key: Original S3 key
            - email_id: None (not available when using direct S3 keys)
            - sender: Extracted from S3 key path if possible
            - subject: None (not available when using direct S3 keys)
            - email_date: None (not available when using direct S3 keys)

    Example:
        >>> keys = [
        ...     "invoices/billing_at_ysplumbing.com/2024-01/invoice1.pdf",
        ...     "invoices/bmwpsupply_at_gmail.com/2023-02/invoice2.pdf"
        ... ]
        >>> pdfs = get_pdfs_by_keys(keys)
        >>> print(f"Downloaded {len(pdfs)} PDFs")
    """
    results = []

    for s3_key in s3_keys:
        if not s3_key or not s3_key.lower().endswith('.pdf'):
            print(f"Skipping non-PDF key: {s3_key}")
            continue

        print(f"Processing: {s3_key}...", end=" ")

        # Download if needed
        local_path = download_pdf_if_needed(s3_key, cache_dir)

        if local_path is None:
            print("FAILED")
            continue

        print("OK")

        # Try to extract sender from S3 key path
        # Expected format: invoices/{sender_email}/YYYY-MM/{filename}.pdf
        sender = None
        parts = s3_key.split('/')
        if len(parts) >= 2:
            # Convert back from path-safe format (e.g., "billing_at_vendor.com" -> "billing@vendor.com")
            sender = parts[1].replace('_at_', '@')

        results.append({
            'local_path': local_path,
            's3_key': s3_key,
            'email_id': None,
            'sender': sender,
            'subject': None,
            'email_date': None
        })

    print(f"\nSuccessfully retrieved {len(results)} PDFs")
    return results


def get_pdfs_from_local(
    directory: str | Path,
    recursive: bool = False,
    sample_size: int = None,
    seed: int = DEFAULT_SEED
) -> List[Dict[str, Any]]:
    """
    Load PDFs from a local directory for extraction.

    Use this when you have PDFs stored locally (e.g., for testing or
    when working with files not in S3).

    Args:
        directory: Path to the directory containing PDF files.
        recursive: If True, search subdirectories recursively. Default False.
        sample_size: Optional limit on number of PDFs to return. If None,
            returns all PDFs found.
        seed: Random seed for reproducible sampling when sample_size is set.

    Returns:
        List[Dict]: List of dicts with keys:
            - local_path: Path to the local PDF file
            - s3_key: None (local files don't have S3 keys)
            - email_id: None (not available for local files)
            - sender: Extracted from parent directory name if possible
            - subject: None (not available for local files)
            - email_date: None (not available for local files)

    Examples:
        >>> # Load all PDFs from a directory
        >>> pdfs = get_pdfs_from_local("/path/to/invoices")
        >>> results_df = run_experiment(pdfs=pdfs)

        >>> # Load recursively with sampling
        >>> pdfs = get_pdfs_from_local("/path/to/invoices", recursive=True, sample_size=10)

        >>> # Load from relative path
        >>> pdfs = get_pdfs_from_local("../../test_invoices")
    """
    directory = Path(directory)

    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")

    if not directory.is_dir():
        raise NotADirectoryError(f"Path is not a directory: {directory}")

    # Find PDF files
    if recursive:
        pdf_files = list(directory.rglob("*.pdf")) + list(directory.rglob("*.PDF"))
    else:
        pdf_files = list(directory.glob("*.pdf")) + list(directory.glob("*.PDF"))

    # Remove duplicates (in case of case-insensitive filesystem)
    pdf_files = list(set(pdf_files))
    pdf_files.sort()  # Sort for consistent ordering

    if not pdf_files:
        print(f"No PDF files found in: {directory}")
        return []

    print(f"Found {len(pdf_files)} PDF files in: {directory}")

    # Apply sampling if requested
    if sample_size is not None and sample_size < len(pdf_files):
        random.seed(seed)
        pdf_files = random.sample(pdf_files, sample_size)
        print(f"Sampled {sample_size} PDFs (seed={seed})")

    results = []

    for pdf_path in pdf_files:
        # Try to extract sender from parent directory name
        # This handles structures like: /invoices/vendor_name/invoice.pdf
        sender = None
        parent_name = pdf_path.parent.name
        if parent_name and parent_name != directory.name:
            # Convert common path-safe formats to email-like format
            sender = parent_name.replace('_at_', '@').replace('_', ' ')

        results.append({
            'local_path': pdf_path.resolve(),
            's3_key': None,
            'email_id': None,
            'sender': sender,
            'subject': None,
            'email_date': None
        })
        print(f"  Added: {pdf_path.name}")

    print(f"\nLoaded {len(results)} PDFs from local directory")
    return results


def get_pdfs(
    sample_size: int = DEFAULT_SAMPLE_SIZE,
    seed: int = DEFAULT_SEED,
    cache_dir: Path = CACHE_DIR,
    ensure_all_vendors: bool = True
) -> List[Dict[str, Any]]:
    """
    Get a random sample of PDFs from the email_download_queue and download from S3.

    This function queries `raw.email_download_queue` for emails with PDFs,
    downloads them from S3 (caching locally), and returns metadata with local paths.

    When ensure_all_vendors=True (default), guarantees at least one PDF from each
    distinct vendor before filling with random samples. This ensures comprehensive
    coverage across all vendors in the scorecard.

    Uses a fixed seed for reproducibility - the same seed will always
    return the same sample of PDFs, making experiments repeatable.

    Args:
        sample_size: Number of PDFs to return. If ensure_all_vendors=True and
            there are more vendors than sample_size, returns one PDF per vendor
            (may exceed sample_size).
        seed: Random seed for reproducible sampling. Default is 42.
        cache_dir: Local directory for caching downloaded PDFs.
        ensure_all_vendors: If True (default), ensure at least one PDF from each
            vendor is included. Set to False for pure random sampling.

    Returns:
        List[Dict]: List of dicts with keys:
            - local_path: Path to the local PDF file
            - s3_key: Original S3 key
            - email_id: email_download_queue UUID
            - sender: email sender (useful for vendor identification)
            - subject: email subject
            - email_date: when the email was received

    Example:
        >>> # Get at least one PDF from each vendor
        >>> pdfs = get_pdfs(sample_size=10, seed=42, ensure_all_vendors=True)
        >>> print(f"Sampled {len(pdfs)} PDFs from {len(set(p['sender'] for p in pdfs))} vendors")
        Sampled 10 PDFs from 8 vendors
    """
    # Get records from database
    db_records = get_pdfs_from_db(
        sample_size=sample_size,
        seed=seed,
        ensure_all_vendors=ensure_all_vendors
    )

    if not db_records:
        print("No PDF records found in database")
        return []

    results = []
    vendors_included = set()

    for record in db_records:
        s3_keys = record.get('s3_keys', [])
        sender = record.get('sender', '')

        # Process each PDF in the email
        for s3_key in s3_keys:
            if not s3_key or not s3_key.lower().endswith('.pdf'):
                continue

            print(f"Processing: {s3_key}...", end=" ")

            # Download if needed
            local_path = download_pdf_if_needed(s3_key, cache_dir)

            if local_path is None:
                print("FAILED")
                continue

            print("OK")

            results.append({
                'local_path': local_path,
                's3_key': s3_key,
                'email_id': str(record['id']),
                'sender': sender,
                'subject': record.get('subject', ''),
                'email_date': record.get('email_date')
            })
            vendors_included.add(sender)

            # For ensure_all_vendors mode, only take one PDF per email record
            # to maximize vendor diversity
            if ensure_all_vendors:
                break

            # Stop if we have enough PDFs (for non-ensure mode)
            if not ensure_all_vendors and len(results) >= sample_size:
                break

        # Stop conditions based on mode
        if ensure_all_vendors:
            # In ensure mode, continue until we have at least sample_size
            # OR we've processed all vendor records
            if len(results) >= sample_size:
                break
        else:
            if len(results) >= sample_size:
                break

    print(f"\nSuccessfully retrieved {len(results)} PDFs from {len(vendors_included)} vendors")
    return results

In [7]:
# Test the function - now ensures all vendors are represented
sample_pdfs = get_pdfs(sample_size=10, seed=42, ensure_all_vendors=True)

# Show vendor coverage
vendors = set(pdf['sender'] for pdf in sample_pdfs)
print(f"\nSampled {len(sample_pdfs)} PDFs from {len(vendors)} vendors:")
print("\nVendor coverage:")
for vendor in sorted(vendors):
    count = sum(1 for pdf in sample_pdfs if pdf['sender'] == vendor)
    print(f"  - {vendor}: {count} PDF(s)")

print("\nPDF details:")
for pdf in sample_pdfs[:5]:  # Show first 5
    print(f"  - {pdf['local_path'].name}")
    print(f"    Sender: {pdf['sender']}")
if len(sample_pdfs) > 5:
    print(f"  ... and {len(sample_pdfs) - 5} more")

Found 9 distinct vendors
Processing: invoices/mendy_at_flowmarksupply.com/2024-09/191bb6c78e6391dd_Invoice_I-0000574-2_from_FLOWMARKPLUMBINGSUPPLY.pdf... OK
Processing: invoices/billing_at_ysplumbing.com/2024-01/18d347c9b80eec90_PH_8553_012224_82993894.PDF... OK
Processing: invoices/billing_at_pyramidps.com/2023-07/18959112537c3647_PH_10_071523_22836883.PDF... OK
Processing: invoices/orders_at_supremeplumbingnj.com/2021-03/178567422b4ca2e3_Invoice_I-0049635.pdf... OK
Processing: invoices/breindy_at_parkplumbingsupply.com/2023-12/18c3c3bedcfbed2b_PH_3_120523_65206918.PDF... OK
Processing: invoices/info_at_wwps.co/2023-07/1891e367181b291d_PH_11_070323_72228809.PDF... OK
Processing: invoices/info_at_wwps.co/2025-02/194d8cafcace25e0_PH_18_020525_72365489.PDF... OK
Processing: invoices/bmwpsupply_at_gmail.com/2022-08/182f667d29b73bfb_PH_3_083122_7254749.PDF... OK
Processing: invoices/accounting_at_supremeplumbingnj.com/2025-06/1979e61a405afeb2_Invoice_I-0138848.pdf... OK
Processing: invoice

In [8]:
def extract_and_validate_single(
    pdf_info: Dict[str, Any],
    system_prompt: str = GEMINI_SYSTEM_PROMPT,
    user_prompt: str = GEMINI_USER_PROMPT_TEMPLATE
) -> dict:
    """
    Extract data from a single PDF using Gemini and validate against PDF content.

    Uses extract_and_validate_data() with only the GEMINI model, which includes
    retry logic and fuzzy correction for improved extraction quality.

    Args:
        pdf_info: Dict with keys from get_pdfs() including 'local_path', 's3_key', etc.
        system_prompt: The system prompt used for extraction (for tracking).
        user_prompt: The user prompt used for extraction (for tracking).

    Returns:
        dict: Results containing:
            - file_name: Name of the PDF file
            - local_path: Path to the local PDF file
            - s3_key: Original S3 key
            - sender: Email sender
            - extraction_success: Whether extraction succeeded
            - header_valid: Whether header validation passed
            - line_items_valid: Whether line items validation passed
            - overall_valid: Whether both validations passed
            - confidence_score: Model's confidence score
            - num_line_items: Number of line items extracted
            - extracted_vendor: Vendor extracted by the model
            - header_details: Detailed header validation results
            - line_items_details: Detailed line items validation results
            - extracted_data: The full extracted data (for debugging)
            - raw_json: Raw JSON string of extracted data
            - system_prompt: System prompt used for this extraction
            - user_prompt: User prompt used for this extraction
            - error: Error message if extraction failed
    """
    from data_extraction import extract_and_validate_data
    from enums import GEMINI_MODEL

    pdf_path = pdf_info['local_path']

    result = {
        "file_name": pdf_path.name,
        "local_path": pdf_path,  # Store for comparison view
        "s3_key": pdf_info.get('s3_key', ''),
        "sender": pdf_info.get('sender', ''),
        "email_id": pdf_info.get('email_id', ''),
        "extraction_success": False,
        "header_valid": False,
        "line_items_valid": False,
        "overall_valid": False,
        "confidence_score": None,
        "num_line_items": 0,
        "extracted_vendor": None,
        "header_details": {},
        "line_items_details": {},
        "extracted_data": None,
        "raw_json": None,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "error": None
    }

    try:
        # Extract and validate data using GEMINI model only
        # This includes retry logic and fuzzy correction
        model_responses = extract_and_validate_data(
            models=[GEMINI_MODEL],
            file_path=str(pdf_path)
        )

        if not model_responses or GEMINI_MODEL not in model_responses:
            result["error"] = "Extraction returned None or no GEMINI result"
            return result

        extracted_data = model_responses[GEMINI_MODEL]

        if not extracted_data:
            result["error"] = "GEMINI extraction returned empty data"
            return result

        result["extraction_success"] = True
        result["extracted_data"] = extracted_data
        result["raw_json"] = json.dumps(extracted_data, indent=2, default=str)
        result["confidence_score"] = extracted_data.get("confidence_score")
        result["num_line_items"] = len(extracted_data.get("line_items", []))
        result["extracted_vendor"] = extracted_data.get("vendor")

        # Map validation results from extract_and_validate_data's underscore-prefixed fields
        result["header_valid"] = extracted_data.get("_header_valid", False)
        result["header_details"] = extracted_data.get("_header_details", {})
        result["line_items_valid"] = extracted_data.get("_line_items_valid", False)
        result["line_items_details"] = extracted_data.get("_line_items_details", {})
        result["overall_valid"] = extracted_data.get("_validation_passed", False)

    except Exception as e:
        result["error"] = str(e)

    return result

In [9]:
def run_experiment(
    pdfs: List[Dict[str, Any]] = None,
    sample_size: int = DEFAULT_SAMPLE_SIZE,
    seed: int = DEFAULT_SEED,
    ensure_all_vendors: bool = True
) -> pd.DataFrame:
    """
    Run extraction experiment on PDFs.
    
    Can be used in two ways:
    1. Provide a list of PDFs directly (from get_pdfs() or get_pdfs_by_keys())
    2. Let it fetch PDFs using get_pdfs() with the given parameters
    
    Args:
        pdfs: Optional list of PDF dicts from get_pdfs() or get_pdfs_by_keys().
            If provided, other parameters are ignored.
        sample_size: Number of PDFs to test (used if pdfs is None).
        seed: Random seed for reproducible sampling (used if pdfs is None).
        ensure_all_vendors: If True, ensure at least one PDF from each vendor
            (used if pdfs is None).
    
    Returns:
        pd.DataFrame: Results DataFrame with one row per PDF.
    
    Examples:
        >>> # Method 1: Let run_experiment fetch PDFs
        >>> results_df = run_experiment(sample_size=10, seed=42)
        
        >>> # Method 2: Provide specific PDFs
        >>> my_pdfs = get_pdfs_by_keys(["invoices/vendor/file.pdf"])
        >>> results_df = run_experiment(pdfs=my_pdfs)
    """
    if pdfs is None:
        pdfs = get_pdfs(sample_size=sample_size, seed=seed, ensure_all_vendors=ensure_all_vendors)
    
    results = []
    
    print(f"\nRunning extraction experiment on {len(pdfs)} PDFs...\n")
    
    for i, pdf_info in enumerate(pdfs, 1):
        start_time = time.perf_counter()
        
        print(f"[{i}/{len(pdfs)}] Extracting {pdf_info['local_path'].name}...", end=" ")
        result = extract_and_validate_single(pdf_info)
        results.append(result)

        end_time = time.perf_counter()
        status = "✓" if result["overall_valid"] else "✗"
        duration = end_time - start_time
        print(f"{status} TIMING: {duration:.4f} seconds")
    
    return pd.DataFrame(results)

In [10]:
def display_scorecard(df: pd.DataFrame) -> None:
    """
    Display a formatted scorecard of experiment results.
    
    Args:
        df: DataFrame with experiment results.
    """
    # Summary statistics
    total = len(df)
    extracted = df["extraction_success"].sum()
    header_valid = df["header_valid"].sum()
    line_items_valid = df["line_items_valid"].sum()
    overall_valid = df["overall_valid"].sum()
    
    print("=" * 60)
    print("EXTRACTION SCORECARD")
    print("=" * 60)
    print(f"\nModel: {GEMINI_MODEL}")
    print(f"Total PDFs tested: {total}")
    print()
    print(f"{'Metric':<30} {'Count':>8} {'Rate':>10}")
    print("-" * 50)
    print(f"{'Extraction Success':<30} {extracted:>8} {extracted/total*100:>9.1f}%")
    print(f"{'Header Validation Pass':<30} {header_valid:>8} {header_valid/total*100:>9.1f}%")
    print(f"{'Line Items Validation Pass':<30} {line_items_valid:>8} {line_items_valid/total*100:>9.1f}%")
    print(f"{'Overall Pass (Both Valid)':<30} {overall_valid:>8} {overall_valid/total*100:>9.1f}%")
    print()
    
    # Per-vendor breakdown (using extracted vendor)
    print("\nResults by Extracted Vendor:")
    print("-" * 50)
    vendor_stats = df.groupby("extracted_vendor").agg({
        "extraction_success": "sum",
        "overall_valid": "sum",
        "file_name": "count"
    }).rename(columns={"file_name": "total"})
    
    for vendor, row in vendor_stats.iterrows():
        vendor_display = vendor if vendor else "(Unknown)"
        if len(vendor_display) > 25:
            vendor_display = vendor_display[:22] + "..."
        pass_rate = row["overall_valid"] / row["total"] * 100 if row["total"] > 0 else 0
        print(f"{vendor_display:<25} {int(row['overall_valid']):>3}/{int(row['total']):<3} ({pass_rate:>5.1f}%)")
    
    print("\n" + "=" * 60)

In [11]:
def display_detailed_results(df: pd.DataFrame) -> None:
    """
    Display detailed results table with color coding.
    
    Args:
        df: DataFrame with experiment results.
    """
    # Select columns for display
    display_cols = [
        "file_name",
        "extracted_vendor",
        "sender",
        "extraction_success",
        "header_valid",
        "line_items_valid",
        "overall_valid",
        "confidence_score",
        "num_line_items"
    ]
    
    display_df = df[display_cols].copy()
    
    # Truncate long strings for display
    display_df["extracted_vendor"] = display_df["extracted_vendor"].apply(
        lambda x: x[:20] + "..." if x and len(str(x)) > 20 else x
    )
    display_df["sender"] = display_df["sender"].apply(
        lambda x: x[:25] + "..." if x and len(str(x)) > 25 else x
    )
    
    # Style the dataframe
    def color_bool(val):
        if val is True:
            return "background-color: #90EE90"  # light green
        elif val is False:
            return "background-color: #FFB6C1"  # light red
        return ""
    
    bool_cols = ["extraction_success", "header_valid", "line_items_valid", "overall_valid"]
    
    styled = display_df.style.applymap(
        color_bool,
        subset=bool_cols
    )
    
    display(styled)

In [12]:
# PDF rendering for comparison view
try:
    from pdf2image import convert_from_path
    PDF2IMAGE_AVAILABLE = True
except ImportError:
    PDF2IMAGE_AVAILABLE = False
    print("pdf2image not available. Install with: pip install pdf2image")
    print("Also requires poppler: sudo apt-get install poppler-utils")

import base64
from io import BytesIO

def render_pdf_page(pdf_path: Path, page_num: int = 0, width: int = 500) -> str:
    """
    Render a PDF page as a base64-encoded image for HTML display.
    
    Args:
        pdf_path: Path to the PDF file
        page_num: Page number to render (0-indexed)
        width: Width of the rendered image in pixels
    
    Returns:
        str: Base64-encoded PNG image data URL, or placeholder if rendering fails
    """
    if not PDF2IMAGE_AVAILABLE:
        return None
    
    try:
        pages = convert_from_path(str(pdf_path), first_page=page_num+1, last_page=page_num+1, size=(width, None))
        if pages:
            img = pages[0]
            buffer = BytesIO()
            img.save(buffer, format='PNG')
            img_base64 = base64.b64encode(buffer.getvalue()).decode()
            return f"data:image/png;base64,{img_base64}"
    except Exception as e:
        print(f"Error rendering PDF: {e}")
    return None


def display_comparison(results_df: pd.DataFrame, show_prompts: bool = True) -> None:
    """
    Display an interactive comparison view of extraction results.
    
    Shows the system/user prompts, then for each PDF:
    - Filename and validation status
    - PDF preview (left) and extracted data (right) side by side
    
    Args:
        results_df: DataFrame with extraction results from run_experiment()
        show_prompts: Whether to display the prompts at the top (default True)
    """
    from IPython.display import display, HTML
    
    # Display prompts
    if show_prompts:
        prompts_html = f"""
        <div style="margin-bottom: 30px; font-family: monospace;">
            <details>
                <summary style="cursor: pointer; font-weight: bold; font-size: 16px; padding: 10px; background: #f0f0f0; border-radius: 5px;">
                    [SYSTEM] System Prompt ({len(GEMINI_SYSTEM_PROMPT)} chars) - Click to expand
                </summary>
                <pre style="background: #1e1e1e; color: #d4d4d4; padding: 15px; border-radius: 5px; overflow-x: auto; white-space: pre-wrap; margin-top: 10px; font-size: 12px;">{GEMINI_SYSTEM_PROMPT}</pre>
            </details>
            <details style="margin-top: 15px;">
                <summary style="cursor: pointer; font-weight: bold; font-size: 16px; padding: 10px; background: #f0f0f0; border-radius: 5px;">
                    [USER] User Prompt Template ({len(GEMINI_USER_PROMPT_TEMPLATE)} chars) - Click to expand
                </summary>
                <pre style="background: #1e1e1e; color: #d4d4d4; padding: 15px; border-radius: 5px; overflow-x: auto; white-space: pre-wrap; margin-top: 10px; font-size: 12px;">{GEMINI_USER_PROMPT_TEMPLATE}</pre>
            </details>
        </div>
        <hr style="margin: 30px 0; border: 1px solid #ccc;">
        """
        display(HTML(prompts_html))
    
    # Display each result
    for idx, row in results_df.iterrows():
        # Status indicator (using text instead of emojis for PDF compatibility)
        if row['overall_valid']:
            status = "[PASS]"
            status_color = "#28a745"
        else:
            status = "[FAIL]"
            status_color = "#dc3545"
        
        # Header validation status
        header_status = "OK" if row['header_valid'] else "X"
        line_items_status = "OK" if row['line_items_valid'] else "X"
        
        # Get PDF image
        pdf_path = None
        for col in ['local_path', 'pdf_path']:
            if col in row and row[col]:
                pdf_path = Path(row[col]) if not isinstance(row[col], Path) else row[col]
                break
        
        # Try to find PDF path from the original data if not in row
        if pdf_path is None and 'file_name' in row:
            # Check temp_pdfs directory
            potential_path = CACHE_DIR / row['file_name']
            if potential_path.exists():
                pdf_path = potential_path
        
        # Render PDF
        pdf_img_src = None
        if pdf_path and pdf_path.exists():
            pdf_img_src = render_pdf_page(pdf_path)
        
        # Format extracted data for display
        extracted_data = row.get('extracted_data', {})
        if extracted_data:
            # Format JSON with syntax highlighting
            json_str = json.dumps(extracted_data, indent=2, default=str)
        else:
            json_str = "No data extracted"
        
        # Build HTML
        pdf_html = ""
        if pdf_img_src:
            pdf_html = f'<img src="{pdf_img_src}" style="max-width: 100%; border: 1px solid #ccc; border-radius: 5px;">'
        elif pdf_path:
            pdf_html = f'<div style="padding: 20px; background: #f8f8f8; border-radius: 5px; text-align: center;">PDF: {pdf_path.name}<br><small>(Install pdf2image to preview)</small></div>'
        else:
            pdf_html = '<div style="padding: 20px; background: #f8f8f8; border-radius: 5px; text-align: center;">PDF not found</div>'
        
        item_html = f"""
        <div style="margin-bottom: 40px; border: 2px solid {status_color}; border-radius: 10px; overflow: hidden;">
            <div style="background: {status_color}; color: white; padding: 10px 15px; font-weight: bold;">
                {status} | {row['file_name']}
                <span style="float: right; font-weight: normal;">
                    Header: {header_status} | Line Items: {line_items_status} | 
                    Vendor: {row.get('extracted_vendor', 'N/A')} | 
                    Items: {row.get('num_line_items', 0)}
                </span>
            </div>
            <div style="display: flex; gap: 20px; padding: 15px;">
                <div style="flex: 1; min-width: 0;">
                    <h4 style="margin-top: 0;">PDF Preview</h4>
                    {pdf_html}
                </div>
                <div style="flex: 1; min-width: 0;">
                    <h4 style="margin-top: 0;">Extracted Data</h4>
                    <pre style="background: #1e1e1e; color: #d4d4d4; padding: 15px; border-radius: 5px; overflow: auto; max-height: 600px; font-size: 11px; white-space: pre-wrap;">{json_str}</pre>
                </div>
            </div>
        </div>
        """
        display(HTML(item_html))

print("display_comparison() function loaded")

display_comparison() function loaded


In [13]:
def generate_comparison_html(results_df: pd.DataFrame, show_prompts: bool = True) -> str:
    """
    Generate the full HTML content for comparison view (without displaying).
    
    Args:
        results_df: DataFrame with extraction results from run_experiment()
        show_prompts: Whether to include the prompts at the top (default True)
    
    Returns:
        str: Complete HTML document as a string
    """
    html_parts = []
    
    # HTML document header with print-friendly styles
    html_parts.append("""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Invoice Extraction Comparison Report</title>
        <style>
            @page {
                size: A4 landscape;
                margin: 1cm;
            }
            body {
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                font-size: 12px;
                line-height: 1.4;
                color: #333;
                max-width: 100%;
                margin: 0 auto;
                padding: 20px;
            }
            .prompts-section {
                margin-bottom: 30px;
                font-family: monospace;
            }
            .prompt-box {
                margin-bottom: 20px;
            }
            .prompt-title {
                font-weight: bold;
                font-size: 14px;
                padding: 10px;
                background: #f0f0f0;
                border-radius: 5px;
                margin-bottom: 10px;
            }
            .prompt-content {
                background: #1e1e1e;
                color: #d4d4d4;
                padding: 15px;
                border-radius: 5px;
                overflow-x: auto;
                white-space: pre-wrap;
                font-size: 10px;
            }
            .result-item {
                margin-bottom: 30px;
                border: 2px solid;
                border-radius: 10px;
                overflow: hidden;
                page-break-inside: avoid;
            }
            .result-header {
                color: white;
                padding: 10px 15px;
                font-weight: bold;
            }
            .result-header-passed { background: #28a745; border-color: #28a745; }
            .result-header-failed { background: #dc3545; border-color: #dc3545; }
            .result-body {
                display: flex;
                gap: 20px;
                padding: 15px;
            }
            .result-column {
                flex: 1;
                min-width: 0;
            }
            .result-column h4 {
                margin-top: 0;
                margin-bottom: 10px;
            }
            .pdf-preview img {
                max-width: 100%;
                border: 1px solid #ccc;
                border-radius: 5px;
            }
            .json-content {
                background: #1e1e1e;
                color: #d4d4d4;
                padding: 15px;
                border-radius: 5px;
                overflow: auto;
                max-height: 500px;
                font-size: 9px;
                white-space: pre-wrap;
                font-family: monospace;
            }
            .status-info {
                float: right;
                font-weight: normal;
            }
            hr {
                margin: 30px 0;
                border: 1px solid #ccc;
            }
            .summary {
                background: #f8f9fa;
                padding: 15px;
                border-radius: 5px;
                margin-bottom: 20px;
            }
        </style>
    </head>
    <body>
    <h1>Invoice Extraction Comparison Report</h1>
    """)
    
    # Summary section
    total = len(results_df)
    passed = results_df["overall_valid"].sum()
    html_parts.append(f"""
    <div class="summary">
        <h3>Summary</h3>
        <p><strong>Total PDFs:</strong> {total} | <strong>Passed:</strong> {passed} | <strong>Failed:</strong> {total - passed} | <strong>Pass Rate:</strong> {passed/total*100:.1f}%</p>
    </div>
    """)
    
    # Prompts section (using text instead of emojis for PDF compatibility)
    if show_prompts:
        html_parts.append(f"""
        <div class="prompts-section">
            <div class="prompt-box">
                <div class="prompt-title">[SYSTEM] System Prompt ({len(GEMINI_SYSTEM_PROMPT)} chars)</div>
                <pre class="prompt-content">{GEMINI_SYSTEM_PROMPT}</pre>
            </div>
            <div class="prompt-box">
                <div class="prompt-title">[USER] User Prompt Template ({len(GEMINI_USER_PROMPT_TEMPLATE)} chars)</div>
                <pre class="prompt-content">{GEMINI_USER_PROMPT_TEMPLATE}</pre>
            </div>
        </div>
        <hr>
        """)
    
    # Each result (using text instead of emojis for PDF compatibility)
    for idx, row in results_df.iterrows():
        if row['overall_valid']:
            status = "[PASS]"
            status_class = "result-header-passed"
            border_color = "#28a745"
        else:
            status = "[FAIL]"
            status_class = "result-header-failed"
            border_color = "#dc3545"
        
        header_status = "OK" if row['header_valid'] else "X"
        line_items_status = "OK" if row['line_items_valid'] else "X"
        
        # Get PDF image
        pdf_path = None
        for col in ['local_path', 'pdf_path']:
            if col in row and row[col]:
                pdf_path = Path(row[col]) if not isinstance(row[col], Path) else row[col]
                break
        
        if pdf_path is None and 'file_name' in row:
            potential_path = CACHE_DIR / row['file_name']
            if potential_path.exists():
                pdf_path = potential_path
        
        # Render PDF
        pdf_html = ""
        if pdf_path and pdf_path.exists():
            pdf_img_src = render_pdf_page(pdf_path, width=400)
            if pdf_img_src:
                pdf_html = f'<img src="{pdf_img_src}" style="max-width: 100%;">'
            else:
                pdf_html = f'<div style="padding: 20px; background: #f8f8f8;">PDF: {pdf_path.name}</div>'
        else:
            pdf_html = '<div style="padding: 20px; background: #f8f8f8;">PDF not found</div>'
        
        # Format JSON
        extracted_data = row.get('extracted_data', {})
        json_str = json.dumps(extracted_data, indent=2, default=str) if extracted_data else "No data extracted"
        
        html_parts.append(f"""
        <div class="result-item" style="border-color: {border_color};">
            <div class="result-header {status_class}">
                {status} | {row['file_name']}
                <span class="status-info">
                    Header: {header_status} | Line Items: {line_items_status} | 
                    Vendor: {row.get('extracted_vendor', 'N/A')} | 
                    Items: {row.get('num_line_items', 0)}
                </span>
            </div>
            <div class="result-body">
                <div class="result-column pdf-preview">
                    <h4>PDF Preview</h4>
                    {pdf_html}
                </div>
                <div class="result-column">
                    <h4>Extracted Data</h4>
                    <pre class="json-content">{json_str}</pre>
                </div>
            </div>
        </div>
        """)
    
    html_parts.append("</body></html>")
    return "".join(html_parts)


def save_comparison_to_pdf(
    results_df: pd.DataFrame,
    output_path: str | Path = "output/comparison_report.pdf",
    show_prompts: bool = True
) -> Path:
    """
    Save the comparison view to a PDF file.
    
    Requires weasyprint: pip install weasyprint
    
    Args:
        results_df: DataFrame with extraction results from run_experiment()
        output_path: Path to save the PDF file (default: output/comparison_report.pdf)
        show_prompts: Whether to include the prompts in the PDF (default True)
    
    Returns:
        Path: Path to the saved PDF file
    
    Example:
        >>> results_df = run_experiment(pdfs=pdfs)
        >>> pdf_path = save_comparison_to_pdf(results_df, "my_report.pdf")
        >>> print(f"Saved to: {pdf_path}")
    """
    try:
        from weasyprint import HTML as WeasyHTML
    except ImportError:
        print("weasyprint not installed. Install with: pip install weasyprint")
        print("On Fedora/RHEL: sudo dnf install weasyprint")
        print("On Ubuntu/Debian: sudo apt-get install weasyprint")
        return None
    
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"Generating HTML content...")
    html_content = generate_comparison_html(results_df, show_prompts=show_prompts)
    
    print(f"Converting to PDF...")
    WeasyHTML(string=html_content).write_pdf(output_path)
    
    print(f"PDF saved to: {output_path.resolve()}")
    return output_path


def save_comparison_to_html(
    results_df: pd.DataFrame,
    output_path: str | Path = "output/comparison_report.html",
    show_prompts: bool = True
) -> Path:
    """
    Save the comparison view to an HTML file (can be opened in browser and printed to PDF).
    
    Args:
        results_df: DataFrame with extraction results from run_experiment()
        output_path: Path to save the HTML file
        show_prompts: Whether to include the prompts (default True)
    
    Returns:
        Path: Path to the saved HTML file
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    html_content = generate_comparison_html(results_df, show_prompts=show_prompts)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML saved to: {output_path.resolve()}")
    print(f"  Open in browser and use Print > Save as PDF if weasyprint is not available")
    return output_path


print("PDF export functions loaded:")
print("  - save_comparison_to_pdf(results_df, 'output/report.pdf')")
print("  - save_comparison_to_html(results_df, 'output/report.html')")

PDF export functions loaded:
  - save_comparison_to_pdf(results_df, 'output/report.pdf')
  - save_comparison_to_html(results_df, 'output/report.html')


## Run Experiment

Execute the extraction experiment on a sample of PDFs from the database.

## Run the experiments with local PDFs
When you need to download specific PDFs and test

In [14]:
# Load PDFs from a local directory
# pdfs = get_pdfs_from_local("/var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/invoices/KnownHasIssue")
pdfs = get_pdfs_from_local("/var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/invoices/KnownHasIssue/Batch100")
results_df = run_experiment(pdfs=pdfs)

Found 100 PDF files in: /var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/invoices/KnownHasIssue/Batch100
  Added: 16e90a9698f22188_PH_4_112119_72647818.pdf
  Added: 17e7a2a42be8de63_PH_11_012022_7229277.pdf
  Added: 17f4d4f294f04eb5_PH_18_030222_72303927.pdf
  Added: 19094d5dc7f15091_PH_8_070824_72729517.pdf
  Added: 190bdb8af0dba3c2_Invoice_I-0000360_from_FLOWMARKPLUMBINGSUPPLY.pdf
  Added: 1965acf5915d2cd2_doc00030844.pdf
  Added: 196651b43b62cce9_PH_7_042325_72203925.PDF
  Added: 1967f81cc716c39e_PH_6105_042825_83130682.PDF
  Added: 19684041bce430b8_doc00032186.pdf
  Added: 1968efa93de56644_PH_3293_050125_83504283.PDF
  Added: 196914b29725e432_PH_14_031925_72241783.PDF
  Added: 196914b29725e432_PH_14_032525_72278856.PDF
  Added: 196914b29725e432_PH_14_032525_72301844.PDF
  Added: 196914b29725e432_PH_20_032425_72205639.PDF
  Added: 196a67bb9cb42c45_Invoice_I-0005331.pdf
  Added: 196a8124e702984d_doc00033466.pdf
  Added: 196add86584a8dcc_PH_6541_050725

/var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/scripts/backfill_invoices_data/data_extraction.py:369: UserWarning: 
[PROMPT VERSION MISMATCH]
Schema content changed but version not updated.
Current hash:  61a8e6ecb9d3
Known hash:    c35eef19bb92
Actions: 
1) Update PROMPT_VERSION
2) Set KNOWN_HASH = '61a8e6ecb9d3'
3) Update PROMPT_CHANGELOG.md with the change
  "prompt_hash": get_prompt_metadata().get("prompt_hash", None)


✓ TIMING: 48.2015 seconds
[2/100] Extracting 17e7a2a42be8de63_PH_11_012022_7229277.pdf... ✓ TIMING: 21.9127 seconds
[3/100] Extracting 17f4d4f294f04eb5_PH_18_030222_72303927.pdf... ✓ TIMING: 52.0442 seconds
[4/100] Extracting 19094d5dc7f15091_PH_8_070824_72729517.pdf... ✓ TIMING: 31.4193 seconds
[5/100] Extracting 190bdb8af0dba3c2_Invoice_I-0000360_from_FLOWMARKPLUMBINGSUPPLY.pdf... ✓ TIMING: 28.4562 seconds
[6/100] Extracting 1965acf5915d2cd2_doc00030844.pdf... ✓ TIMING: 38.7006 seconds
[7/100] Extracting 196651b43b62cce9_PH_7_042325_72203925.PDF... ✓ TIMING: 42.8769 seconds
[8/100] Extracting 1967f81cc716c39e_PH_6105_042825_83130682.PDF... ✓ TIMING: 32.8676 seconds
[9/100] Extracting 19684041bce430b8_doc00032186.pdf... ✓ TIMING: 26.5934 seconds
[10/100] Extracting 1968efa93de56644_PH_3293_050125_83504283.PDF... ✓ TIMING: 26.8948 seconds
[11/100] Extracting 196914b29725e432_PH_14_031925_72241783.PDF... ✓ TIMING: 24.5764 seconds
[12/100] Extracting 196914b29725e432_PH_14_032525_7227885

## Run the experiment sampling from the database
When you want to run the prompt against a random sample

In [42]:
# Configure experiment
SAMPLE_SIZE = 10  # Will get at least this many, ensuring all vendors are covered
SEED = 42
ENSURE_ALL_VENDORS = True  # Set to False for pure random sampling

# Run the experiment
# results_df = run_experiment(
#     sample_size=SAMPLE_SIZE,
#     seed=SEED,
#     ensure_all_vendors=ENSURE_ALL_VENDORS
# )

# Scorecard summary

In [15]:
# Display scorecard summary
display_scorecard(results_df)

EXTRACTION SCORECARD

Model: gemini-2.5-pro
Total PDFs tested: 100

Metric                            Count       Rate
--------------------------------------------------
Extraction Success                   98      98.0%
Header Validation Pass               98      98.0%
Line Items Validation Pass           98      98.0%
Overall Pass (Both Valid)            98      98.0%


Results by Extracted Vendor:
--------------------------------------------------
BMW Plumbing Supplies      23/23  (100.0%)
Flow Mark Plumbing Supply  12/12  (100.0%)
Supreme Plumbing Supply     3/3   (100.0%)
World Wide Plumbing        48/48  (100.0%)
Y & S Plumbing Supplies    12/12  (100.0%)



In [16]:
# Display detailed results table
display_detailed_results(results_df)

/tmp/ipykernel_1012843/2057545943.py:41: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = display_df.style.applymap(


,file_name,extracted_vendor,sender,extraction_success,header_valid,line_items_valid,overall_valid,confidence_score,num_line_items
0,16e90a9698f22188_PH_4_112119_72647818.pdf,World Wide Plumbing,None,True,True,True,True,0.95,40
1,17e7a2a42be8de63_PH_11_012022_7229277.pdf,World Wide Plumbing,None,True,True,True,True,0.95,2
2,17f4d4f294f04eb5_PH_18_030222_72303927.pdf,World Wide Plumbing,None,True,True,True,True,0.95,2
3,19094d5dc7f15091_PH_8_070824_72729517.pdf,BMW Plumbing Supplie...,None,True,True,True,True,0.99,2
4,190bdb8af0dba3c2_Invoice_I-0000360_from_FLOWMARKPLUMBINGSUPPLY.pdf,Flow Mark Plumbing S...,None,True,True,True,True,0.99,6
5,1965acf5915d2cd2_doc00030844.pdf,BMW Plumbing Supplie...,None,True,True,True,True,0.95,11
6,196651b43b62cce9_PH_7_042325_72203925.PDF,World Wide Plumbing,None,True,True,True,True,0.95,35
7,1967f81cc716c39e_PH_6105_042825_83130682.PDF,Y & S Plumbing Suppl...,None,True,True,True,True,0.99,7
8,19684041bce430b8_doc00032186.pdf,BMW Plumbing Supplie...,None,True,True,True,True,0.95,9
9,1968efa93de56644_PH_3293_050125_83504283.PDF,Y & S Plumbing Suppl...,None,True,True,True,True,0.99,3


# Comparison View

Use `display_comparison(results_df)` to view:
- System and user prompts (collapsible)
- Each PDF side-by-side with its extracted data
- Validation status for each item

In [ ]:
# Display the comparison view
# Set show_prompts=False to hide the prompts section
display_comparison(results_df, show_prompts=True)

## Inspect Failed Extractions

Examine details of any failed extractions for debugging.

In [27]:
# Show failed extractions
failed_df = results_df[~results_df["overall_valid"]]

if len(failed_df) == 0:
    print("All extractions passed validation!")
else:
    print(f"\n{len(failed_df)} Failed Extractions:\n")
    for idx, row in failed_df.iterrows():
        print(f"File: {row['file_name']}")
        print(f"  S3 Key: {row['s3_key']}")
        print(f"  Sender: {row['sender']}")
        print(f"  Extracted Vendor: {row['extracted_vendor']}")
        print(f"  Extraction Success: {row['extraction_success']}")
        print(f"  Header Valid: {row['header_valid']}")
        print(f"  Line Items Valid: {row['line_items_valid']}")
        
        if row['error']:
            print(f"  Error: {row['error']}")
        
        if row['header_details'] and row['header_details'].get('missing_fields'):
            print(f"  Header Missing: {row['header_details']['missing_fields']}")
        
        if row['line_items_details'] and row['line_items_details'].get('missing_fields'):
            print(f"  Line Items Missing: {row['line_items_details']['missing_fields']}")
        
        print()


1 Failed Extractions:

File: invoices_billing_at_ysplumbing.com_2023-05_187df9a79c0adb4b_PH_9929_050223_83648333.PDF
  S3 Key: invoices/billing_at_ysplumbing.com/2023-05/187df9a79c0adb4b_PH_9929_050223_83648333.PDF
  Sender: billing@ysplumbing.com
  Extracted Vendor: Y & S Plumbing Supplies
  Extraction Success: True
  Header Valid: True
  Line Items Valid: False
  Line Items Missing: ['qty (item 0) - not found in PDF', 'qty (item 1) - not found in PDF']



## Export Results

Save results to CSV for further analysis.

In [45]:
# Export to CSV (excluding large extracted_data column but including raw_json and prompts)
export_cols = [
    "file_name", "s3_key", "sender", "email_id", "extracted_vendor",
    "extraction_success", "header_valid", "line_items_valid", "overall_valid",
    "confidence_score", "num_line_items", "error",
    "raw_json", "system_prompt", "user_prompt"
]

output_path = Path("output/scorecard_results_2.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

results_df[export_cols].to_csv(output_path, index=False)
print(f"Results exported to: {output_path.resolve()}")

# Also export just the raw JSON responses to a separate JSONL file for easier parsing
jsonl_path = Path("output/scorecard_extractions_2.jsonl")
with open(jsonl_path, 'w') as f:
    for idx, row in results_df.iterrows():
        record = {
            "file_name": row["file_name"],
            "s3_key": row["s3_key"],
            "extraction_success": row["extraction_success"],
            "overall_valid": row["overall_valid"],
            "extracted_data": row["extracted_data"]
        }
        f.write(json.dumps(record, default=str) + "\n")
print(f"Raw extractions exported to: {jsonl_path.resolve()}")

Results exported to: /var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/scripts/backfill_invoices_data/output/scorecard_results_2.csv
Raw extractions exported to: /var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/scripts/backfill_invoices_data/output/scorecard_extractions_2.jsonl


In [17]:
save_comparison_to_pdf(results_df, "output/comparison_report_pro_batch_100.pdf", show_prompts=True)

Generating HTML content...
Converting to PDF...


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.009s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

PDF saved to: /var/home/ej/Workspace/MPI-Plumbing/InvoiceProject/BackendCode/explore-new-prompt/scripts/backfill_invoices_data/output/comparison_report_pro_batch_100.pdf


PosixPath('output/comparison_report_pro_batch_100.pdf')

## Cleanup

Optionally clear the PDF cache.

In [ ]:
# Uncomment to clear the PDF cache
# import shutil
# if CACHE_DIR.exists():
#     shutil.rmtree(CACHE_DIR)
#     print(f"Cleared cache: {CACHE_DIR}")